# Сравнение на основе сохраненных метрик (JSON)


In [5]:
import pandas as pd
import plotly.express as px
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "README.md").exists()
    and (path / "Data_making").is_dir()
    and (path / "Models").is_dir()
)

In [6]:
import json

base_path = PROJECT_ROOT / "Models" / "Compare models"
logreg_path = base_path / 'logreg_metrics.json'
cat_path = base_path / 'catboost_metrics.json'

logreg_metrics = json.loads(logreg_path.read_text())
cat_metrics = json.loads(cat_path.read_text())

metrics_df = pd.DataFrame([logreg_metrics, cat_metrics])
metrics_df['Feature Scenario'] = metrics_df['feature_scenario']
metrics_df['N Features'] = metrics_df['feature_columns'].apply(len)
metrics_df

,model,feature_scenario,feature_columns,excluded_leakage_risk_cols,split_strategy,random_state,train_size,validation_size,test_size,threshold,...,test_FN,test_TP,CV_AUC_mean,CV_AUC_std,CV_AUC_best,best_params,best_iteration,reproducibility,Feature Scenario,N Features
0,LogReg,common_no_avg_grade,"[Класс, Район, Часы_самоподготовки_в_неделю, П...",[Средний_балл],stratified_train_valid_test,42,915,305,305,0.535,...,61,129,0.761999,0.025011,NaN,NaN,NaN,NaN,common_no_avg_grade,12
1,CatBoost,common_no_avg_grade,"[Класс, Район, Часы_самоподготовки_в_неделю, П...",[Средний_балл],stratified_train_valid_test,42,915,305,305,0.500,...,59,131,NaN,NaN,0.75914,"{'bagging_temperature': 0, 'depth': 4, 'iterat...",5.0,"{'numpy_rng': 'np.random.default_rng(42)', 'ca...",common_no_avg_grade,12


In [7]:
# Визуализация сохраненных метрик (устойчиво к разным ключам JSON)
# Берем только общие и числовые метрики для честного сравнения.
metric_order = [
    'test_ROC_AUC',
    'test_PR_AUC',
    'test_Accuracy',
    'test_Precision',
    'test_Recall',
    'test_F1',
]

available_metrics = [
    m for m in metric_order
    if m in metrics_df.columns and pd.api.types.is_numeric_dtype(metrics_df[m])
]

if not available_metrics:
    raise ValueError('Не найдены общие числовые метрики для сравнения.')

metrics_plot_df = metrics_df[['model'] + available_metrics].copy()
metrics_long = metrics_plot_df.melt(id_vars='model', var_name='metric', value_name='value')

fig = px.bar(
    metrics_long,
    x='metric', y='value', color='model', barmode='group', text='value',
    category_orders={'metric': available_metrics},
    title='Сравнение ключевых метрик моделей'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0, 1]))
fig.show()

# Условия оценки показываются отдельно от финальных test-метрик.
evaluation_context_cols = [
    'model', 'threshold', 'threshold_strategy', 'threshold_fallback_used',
    'Feature Scenario', 'N Features', 'split_strategy',
]
display(metrics_df[evaluation_context_cols])


,model,threshold,threshold_strategy,threshold_fallback_used,Feature Scenario,N Features,split_strategy
0,LogReg,0.535,max_recall_at_precision_ge_0.80,False,common_no_avg_grade,12,stratified_train_valid_test
1,CatBoost,0.500,max_recall_at_precision_ge_0.80,False,common_no_avg_grade,12,stratified_train_valid_test


In [8]:
# Основная сравнительная таблица независимых test-метрик.
comparison_table = metrics_df[
    [
        'model', 'Feature Scenario', 'N Features', 'split_strategy',
        'threshold', 'threshold_strategy', 'threshold_fallback_used',
    ] + available_metrics
].copy()
comparison_table


,model,Feature Scenario,N Features,split_strategy,threshold,threshold_strategy,threshold_fallback_used,test_ROC_AUC,test_PR_AUC,test_Accuracy,test_Precision,test_Recall,test_F1
0,LogReg,common_no_avg_grade,12,stratified_train_valid_test,0.535,max_recall_at_precision_ge_0.80,False,0.825034,0.880579,0.731148,0.860000,0.678947,0.758824
1,CatBoost,common_no_avg_grade,12,stratified_train_valid_test,0.500,max_recall_at_precision_ge_0.80,False,0.804874,0.861234,0.731148,0.850649,0.689474,0.761628


## Вывод

Сравнение является корректным: обе модели обучались и оценивались на одном разбиении `stratified_train_valid_test`, используют одинаковые 12 признаков без `Средний_балл`, а пороги подбирались только на validation-выборке. Финальные метрики рассчитаны на общей независимой test-выборке из 305 наблюдений.

**По целевому правилу выбора модели CatBoost формально лучше.** Задача порога — максимизировать Recall при Precision не ниже 0.80. На test обе модели сохраняют требуемый уровень Precision, но CatBoost показывает Recall 0.6895 против 0.6789 у Logistic Regression (+1.05 п.п.) и F1 0.7616 против 0.7588 (+0.28 п.п.). В абсолютных значениях CatBoost находит 131 положительный случай вместо 129 и допускает 59 пропусков вместо 61. За это он дает 23 ложноположительных прогноза вместо 21. Accuracy у моделей одинаковая — 0.7311.

**При этом Logistic Regression лучше ранжирует объекты и точнее оценивает вероятности:** ROC AUC выше на 2.02 п.п. (0.8250 против 0.8049), PR AUC — на 1.93 п.п. (0.8806 против 0.8612), Precision — на 0.94 п.п. (0.8600 против 0.8506), а LogLoss ниже (0.5226 против 0.6667; меньше — лучше). Поэтому Logistic Regression предпочтительнее, если важны качество ранжирования, вероятностный прогноз, интерпретируемость или снижение числа ложных срабатываний.

**Итоговая рекомендация:** для зафиксированной в проекте цели — найти максимум положительных случаев при Precision ≥ 0.80 — выбрать **CatBoost**. Однако его преимущество невелико: всего 2 дополнительных TP ценой 2 дополнительных FP на одной test-выборке. Поэтому результат нельзя трактовать как доказательство общего превосходства CatBoost; перед внедрением устойчивость различий следует проверить с помощью доверительных интервалов или повторной кросс-валидации.